# Computing Tempered Energy 

We aim to implement a parallel tempering scheme for annealed MCMC as per Du et al.'s [2024 paper](https://arxiv.org/pdf/2302.11552) on compositional generation with energy-based diffusion. In order to implement parallel tempering into the sampling regime of annealed MCMC, we aim to evaluate the difference between true tempered energy and our approximations.

Training follows standard DDPM noise-matching objective


Sampling follows annealed MCMC

Input:
- Transition kernels $k_t(· | ·)$
- Initial distribution $p_T(·)$
- Number of steps $N$

Sampling steps:
- Initialize $x_T \sim p_T(·)$
- for $t = T, T-1, ... ,  0$ do
    - for $i = 1, ... ,  N$ do
        - $x_t \sim k_t(· | x_t)$
    - end for
    - $x_{t-1} = x_t$
- end for

For our transition kernel, we choose ULA algorithm
$$q(x_t\mid x_{t-1}) = N(x_{t-1} + \frac{\sigma_L^2}{2} \nabla_x f(x_{t-1}), \sigma_L^2 I)$$
We choose step size $\beta_t$
$$x_{t+1} = x_t - \beta_t \nabla_x p_t(x) + \sqrt{2} \beta_t z$$
where
$$\nabla \text{model output} = \nabla_x p_t(x) = \frac{\epsilon_{\theta}(x_t, t)}{\sqrt{1-\bar{\alpha}_t}}$$
See appendix b.2 of reduce reuse recycle and section 2.2

### Setup

In [1]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- hyperparams ----
mu_0, sigma_0 = 0.0, 1.0
n_diffusion_steps = 100

n_steps = 15_000
batch_size = 128
lr = 3e-4

train_dataset_size = 50_000
n_samples = 50_000

n_langevin_steps = 10
step_scale = 0.6

ckpt_dir = "model_checkpoints"

### Schedule

In [2]:
def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    t = torch.linspace(0, timesteps, steps, dtype=torch.float32) / timesteps
    alphas_cumprod = torch.cos((t + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)

betas = cosine_beta_schedule(n_diffusion_steps).to(device)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)  # alphā_t
# sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)

ts_desc = torch.arange(n_diffusion_steps - 1, -1, -1, device=device)

### Model

In [3]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        if t.dim() == 2:
            t = t.squeeze(-1)

        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / half
        )
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))

        return emb


class MLP(nn.Module):
    def __init__(
        self,
        x_dim=1,
        hidden_dim=128,
        time_dim=32,
        n_layers=4,
    ):
        super().__init__()

        self.time_embed = SinusoidalTimeEmbedding(time_dim)

        self.input = nn.Linear(x_dim + time_dim, hidden_dim)

        self.layers = nn.ModuleList(
            [nn.Linear(hidden_dim, hidden_dim) for _ in range(n_layers)]
        )

        self.output = nn.Linear(hidden_dim, x_dim)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        h = torch.cat([x, t_emb], dim=-1)

        h = self.input(h)
        h = F.silu(h)

        for layer in self.layers:
            h = h + F.silu(layer(h))

        return self.output(h)


### Train Model

In [4]:
def train_model(temp, save_path, log_every=1000):
    # training data: N(mu0, sigma0/sqrt(temp))  (keep on CPU; move batches to GPU)
    x0_all = torch.normal(
        mean=mu_0,
        std=sigma_0 / np.sqrt(temp),
        size=(train_dataset_size, 1),
    )
    loader = DataLoader(
        TensorDataset(x0_all),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,      # keep simple; set >0 if you want
        pin_memory=True,    # helps H2D transfer
    )

    model = MLP().to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    model.train()

    data_iter = iter(loader)

    for step in range(1, n_steps + 1):
        # get a batch (restart dataloader when exhausted; still no "epochs" variable)
        try:
            (x0,) = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)  # reshuffles because shuffle=True
            (x0,) = next(data_iter)

        x0 = x0.to(device, non_blocking=True)

        t = torch.randint(0, n_diffusion_steps, (batch_size, 1), device=device)
        a_bar = alpha_bar[t]
        
        noise = torch.randn_like(x0)
        xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * noise

        eps_hat = model(xt, t)
        loss = ((noise - eps_hat) ** 2).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % log_every == 0:
            print(f"temp={temp} step={step} loss={loss.item():.4f}")

    torch.save(model.state_dict(), save_path)
    return model

### Sampling

In [5]:
def tsr_func(t_idx, T_modeling, T_sampling):
    # Keep it close to your original formula
    sigma = sigma_0 / np.sqrt(T_modeling)

    a_bar = alpha_bar[t_idx]
    sigma_t = torch.sqrt(1.0 - a_bar)
    alpha_t = torch.sqrt(a_bar)

    eta_t = (alpha_t**2) / (sigma_t**2)
    num = eta_t * (sigma ** 2) + 1
    den = (eta_t * (sigma ** 2)) / T_sampling + 1
    return num / den

@torch.no_grad()
def ula_sample(model, T_modeling, T_sampling, n_samples=n_samples):
    model.eval()
    x = torch.randn(n_samples, 1, device=device)
    ones = torch.ones(n_samples, 1, device=device)

    for t in ts_desc:
        t_batch = (t * ones).long()

        for _ in range(n_langevin_steps):
            step = betas[t] * step_scale

            eps_hat = model(x, t_batch)
            
            a_bar = alpha_bar[t]
            score_hat = -eps_hat / torch.sqrt(1.0 - a_bar)  # common conversion

            tsr = tsr_func(t, T_modeling, T_sampling)
            noise = torch.randn_like(x)

            x = x + step * score_hat * tsr + noise * torch.sqrt(2.0 * step)

    return x

### Compare and Plot

In [ ]:
def load_model(path):
    m = MLP().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

def compare(temp_target):
    # model trained at temp_target (your "true")
    m_true = load_model(f"{ckpt_dir}/model_T{temp_target:.1f}.pt")
    x_true = ula_sample(m_true, T_modeling=temp_target, T_sampling=1.0)

    # model trained at temp=1 but sampled at temp_target (your "approx")
    m_base = load_model(f"{ckpt_dir}/model_T{1.0:.1f}.pt")
    x_base = ula_sample(m_base, T_modeling=1.0, T_sampling=temp_target)

    # plot vs analytic gaussian
    xlim = 4
    x_axis = np.linspace(-xlim, xlim, 500)
    std = sigma_0 / np.sqrt(temp_target)
    pdf = (1.0 / (np.sqrt(2*np.pi) * std)) * np.exp(-0.5 * ((x_axis - mu_0)/std)**2)

    bins = np.linspace(-xlim, xlim, 100)
    plt.figure(figsize=(5,4))
    _ = plt.hist(x_base.cpu().numpy(), bins=bins, density=True, alpha=0.5, label="samples (model T=1)"). 
    _ = plt.hist(x_true.cpu().numpy(), bins=bins, density=True, alpha=0.5, label=f"samples (model T={temp_target})")
    plt.plot(x_axis, pdf, label="true gaussian pdf")
    plt.title(f"ULA sample distribution at T={temp_target}")
    plt.xlim(-xlim, xlim)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return x_true, x_base

### Implementation

In [ ]:
temperature = 4

save_path = f"{ckpt_dir}/model_T{temperature:.1f}.pt"
print(save_path)
train_model(temperature, save_path)
compare(temperature)

model_checkpoints/model_T4.0.pt
temp=4 step=1000 loss=0.4511
temp=4 step=2000 loss=0.4180
temp=4 step=3000 loss=0.3816
temp=4 step=4000 loss=0.3117
temp=4 step=5000 loss=0.2283
temp=4 step=6000 loss=0.3830
